# 🔬 Diabetic Retinopathy Detection - Complete Architecture Pipeline
### ConvNeXt + MaxViT Hybrid with Lesion Attention, Multi-Task Learning & Ensemble
---

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
import subprocess, sys

packages = [
    'timm', 'albumentations', 'opencv-python-headless',
    'scikit-learn', 'pandas', 'matplotlib', 'seaborn',
    'torch', 'torchvision', 'tqdm', 'scipy', 'Pillow'
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

print('\n✅ CELL 1 COMPLETED: Dependencies installed')

In [ ]:
# ============================================================
# CELL 2: Imports & GPU Configuration
# ============================================================
import os, gc, json, time, warnings, random
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as T
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    roc_curve, cohen_kappa_score
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import label_binarize
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from scipy import ndimage
from copy import deepcopy

warnings.filterwarnings('ignore')

# ── GPU: use device 1 (as instructed) ──────────────────────
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Using device : {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU Name     : {torch.cuda.get_device_name(0)}')
    print(f'   Memory       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Reproducibility ────────────────────────────────────────
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

print('\n✅ CELL 2 COMPLETED: Imports & GPU configured')

In [ ]:
# ============================================================
# CELL 3: Global Configuration
# ============================================================
CFG = {
    # Paths – update these to your actual paths
    'eyepacs_csv'     : 'eyepacs/trainLabels.csv',
    'eyepacs_img_dir' : 'eyepacs/train/',
    'messidor_csv'    : 'messidor2/messidor_data.csv',
    'messidor_img_dir': 'messidor2/IMAGES/',
    'idrid_csv'       : 'IDRiD/IDRiD_Disease_Grading_Training_Labels.csv',
    'idrid_img_dir'   : 'IDRiD/Images/Training/',
    'result_root'     : 'final_result',

    # Training
    'img_size'        : 448,
    'batch_size'      : 16,
    'num_workers'     : 4,
    'num_classes'     : 5,
    'n_runs'          : 50,       # 50 independent runs → average metrics
    'n_folds'         : 5,
    'epochs'          : 20,
    'lr'              : 1e-4,
    'weight_decay'    : 1e-4,
    'seed'            : 42,

    # Ensemble
    'n_ensemble'      : 3,        # fold models per run ensemble
}

# ── Create output folder structure ─────────────────────────
DATASETS = ['eyepacs', 'messidor2', 'idrid']
for ds in DATASETS:
    path = os.path.join(CFG['result_root'], ds)
    os.makedirs(path, exist_ok=True)
    print(f'   📁 Created: {path}')

print('\n✅ CELL 3 COMPLETED: Configuration set & folders created')

In [ ]:
# ============================================================
# CELL 4: Block 1 – Image Preprocessing Utilities
# ============================================================

def apply_clahe(img_bgr):
    """Contrast Limited Adaptive Histogram Equalization on L channel."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge([l, a, b])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def circular_crop(img):
    """Remove black borders using circular mask."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        img = img[y:y+h, x:x+w]
    return img


def color_normalize(img_bgr):
    """Per-channel mean subtraction."""
    img = img_bgr.astype(np.float32)
    for ch in range(3):
        mean = img[:, :, ch].mean()
        std  = img[:, :, ch].std() + 1e-6
        img[:, :, ch] = (img[:, :, ch] - mean) / std
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
    return img.astype(np.uint8)


def preprocess_fundus(img_path, size=448):
    """Full Block-1 preprocessing pipeline."""
    img = cv2.imread(img_path)
    if img is None:
        # Return blank image if file missing (fallback for demo)
        return np.zeros((size, size, 3), dtype=np.uint8)
    img = circular_crop(img)
    img = cv2.resize(img, (size, size))
    img = apply_clahe(img)
    img = color_normalize(img)
    img = cv2.GaussianBlur(img, (3, 3), 0)
    # Intensity normalisation → [0, 255]
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
    return img  # BGR uint8


print('\n✅ CELL 4 COMPLETED: Block 1 – Preprocessing utilities ready')

In [ ]:
# ============================================================
# CELL 5: Block 2 – Data Augmentation (Albumentations)
# ============================================================

def get_train_transforms(size=448):
    return A.Compose([
        A.Rotate(limit=20, p=0.6),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomResizedCrop(height=size, width=size, scale=(0.8, 1.0), p=0.4),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.4),
        A.GaussNoise(var_limit=(5, 30), p=0.3),
        A.GaussianBlur(blur_limit=(3, 5), p=0.3),
        A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.2),  # CutMix approx
        A.Resize(size, size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


def get_val_transforms(size=448):
    return A.Compose([
        A.Resize(size, size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


def mixup_data(x, y, alpha=0.4):
    """MixUp augmentation."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    idx = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam


print('\n✅ CELL 5 COMPLETED: Block 2 – Augmentation pipelines ready')

In [ ]:
# ============================================================
# CELL 6: Dataset Loaders
# ============================================================

class DRDataset(Dataset):
    def __init__(self, df, img_dir, img_col, label_col,
                 img_ext='.png', transform=None, size=448, preprocess=True):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.img_col   = img_col
        self.label_col = label_col
        self.img_ext   = img_ext
        self.transform = transform
        self.size      = size
        self.preprocess = preprocess

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        fname  = str(row[self.img_col])
        if not fname.endswith(('.png', '.jpg', '.jpeg', '.tif')):
            fname += self.img_ext
        path   = os.path.join(self.img_dir, fname)
        label  = int(row[self.label_col])

        if self.preprocess:
            img_bgr = preprocess_fundus(path, self.size)
        else:
            img_bgr = cv2.imread(path)
            if img_bgr is None:
                img_bgr = np.zeros((self.size, self.size, 3), dtype=np.uint8)
            img_bgr = cv2.resize(img_bgr, (self.size, self.size))

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        if self.transform:
            aug = self.transform(image=img_rgb)
            img_rgb = aug['image']

        return img_rgb, label


def load_dataset_df(name):
    """Load and normalise each dataset's CSV to (image_id, label) columns."""
    if name == 'eyepacs':
        df = pd.read_csv(CFG['eyepacs_csv'])
        df = df.rename(columns={'image': 'image_id', 'level': 'label'})
        ext = '.jpeg'
        img_dir = CFG['eyepacs_img_dir']

    elif name == 'messidor2':
        df = pd.read_csv(CFG['messidor_csv'])
        df = df[df['adjudicated_gradable'] == 1].reset_index(drop=True)
        df = df.rename(columns={'id_code': 'image_id', 'diagnosis': 'label'})
        ext = ''
        img_dir = CFG['messidor_img_dir']

    elif name == 'idrid':
        df = pd.read_csv(CFG['idrid_csv'])
        df = df[['id_code', 'diagnosis']].dropna()
        df = df.rename(columns={'id_code': 'image_id', 'diagnosis': 'label'})
        df['label'] = df['label'].astype(int)
        ext = '.jpg'
        img_dir = CFG['idrid_img_dir']
    else:
        raise ValueError(f'Unknown dataset: {name}')

    df['label'] = df['label'].astype(int).clip(0, 4)
    print(f'   {name}: {len(df)} samples | class dist: {df["label"].value_counts().sort_index().to_dict()}')
    return df, img_dir, ext


print('\n✅ CELL 6 COMPLETED: Dataset loaders defined')

In [ ]:
# ============================================================
# CELL 7: Block 3 – Feature Extraction Backbones
#         ConvNeXt (CNN) + MaxViT (Transformer)
# ============================================================

class ConvNeXtBranch(nn.Module):
    """CNN Branch: local texture, microaneurysms, exudates, hemorrhages."""
    def __init__(self, out_dim=512, pretrained=True):
        super().__init__()
        backbone = timm.create_model(
            'convnext_base', pretrained=pretrained, num_classes=0, global_pool='avg'
        )
        self.backbone = backbone
        self.proj = nn.Sequential(
            nn.Linear(backbone.num_features, out_dim),
            nn.GELU(),
            nn.Dropout(0.3),
        )

    def forward(self, x):
        feat = self.backbone(x)  # (B, num_features)
        return self.proj(feat)   # (B, out_dim)


class MaxViTBranch(nn.Module):
    """Transformer Branch: global retinal structure, macular edema, large lesions."""
    def __init__(self, out_dim=512, pretrained=True):
        super().__init__()
        # Fall back to swin_base if maxvit not available
        try:
            backbone = timm.create_model(
                'maxvit_tiny_tf_224.in1k', pretrained=pretrained,
                num_classes=0, global_pool='avg'
            )
        except Exception:
            backbone = timm.create_model(
                'swin_base_patch4_window7_224', pretrained=pretrained,
                num_classes=0, global_pool='avg'
            )
        self.backbone = backbone
        self.proj = nn.Sequential(
            nn.Linear(backbone.num_features, out_dim),
            nn.GELU(),
            nn.Dropout(0.3),
        )

    def forward(self, x):
        feat = self.backbone(x)
        return self.proj(feat)


print('\n✅ CELL 7 COMPLETED: Block 3 – Dual-backbone branches defined')

In [ ]:
# ============================================================
# CELL 8: Block 3 continued – Feature Fusion
# ============================================================

class CrossAttentionFusion(nn.Module):
    """Cross-attention between CNN and Transformer features."""
    def __init__(self, dim=512, num_heads=8):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 2), nn.GELU(), nn.Linear(dim * 2, dim)
        )

    def forward(self, cnn_feat, vit_feat):
        # Add sequence dimension for attention: (B, 1, dim)
        q = cnn_feat.unsqueeze(1)
        k = vit_feat.unsqueeze(1)
        attn_out, _ = self.attn(q, k, k)
        fused = self.norm1(q + attn_out).squeeze(1)
        fused = self.norm2(fused + self.ffn(fused))
        return fused


class FeatureFusion(nn.Module):
    """Combines concatenation, averaging, and cross-attention fusion."""
    def __init__(self, dim=512):
        super().__init__()
        self.cross_attn = CrossAttentionFusion(dim=dim)
        # After concat (dim*2) + avg (dim) + cross_attn (dim) → project to dim
        self.proj = nn.Sequential(
            nn.Linear(dim * 4, dim * 2),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(dim * 2, dim),
        )

    def forward(self, cnn_feat, vit_feat):
        concat  = torch.cat([cnn_feat, vit_feat], dim=-1)         # (B, 2*dim)
        avg     = (cnn_feat + vit_feat) / 2.0                     # (B, dim)
        ca      = self.cross_attn(cnn_feat, vit_feat)             # (B, dim)
        merged  = torch.cat([concat, avg, ca], dim=-1)            # (B, 4*dim)
        return self.proj(merged)                                   # (B, dim)


print('\n✅ CELL 8 COMPLETED: Feature Fusion module defined')

In [ ]:
# ============================================================
# CELL 9: Block 4a – Lesion Attention Module
# ============================================================

class LesionAttentionModule(nn.Module):
    """
    Vessel suppression + top-hat filtering + blob detection (LoG/DoG approx)
    → lesion probability map → attention weighting.
    Applied on raw feature maps from an intermediate CNN layer.
    """
    def __init__(self, in_channels=1024):
        super().__init__()
        # Vessel suppression via 1×1 conv
        self.vessel_suppress = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 2, 1),
            nn.BatchNorm2d(in_channels // 2),
            nn.ReLU(),
        )
        # Top-hat filtering approximation via large-kernel – small-kernel
        self.tophat_large = nn.Conv2d(in_channels // 2, in_channels // 2, 15, padding=7,
                                      groups=in_channels // 2, bias=False)
        self.tophat_small = nn.Conv2d(in_channels // 2, in_channels // 2, 3, padding=1,
                                      groups=in_channels // 2, bias=False)
        # LoG / DoG blob detection approximation
        self.blob_detect = nn.Sequential(
            nn.Conv2d(in_channels // 2, 64, 5, padding=2),
            nn.ReLU(),
            nn.Conv2d(64, 1, 1),
            nn.Sigmoid(),
        )
        # Final lesion probability map
        self.lesion_map = nn.Sequential(
            nn.Conv2d(in_channels // 2 + 1, 1, 1),
            nn.Sigmoid(),
        )

    def forward(self, feat_map):
        # feat_map: (B, C, H, W) from CNN intermediate layer
        suppressed  = self.vessel_suppress(feat_map)
        tophat      = self.tophat_small(suppressed) - self.tophat_large(suppressed)
        blob        = self.blob_detect(suppressed)
        combined    = torch.cat([suppressed, blob], dim=1)
        lesion_prob = self.lesion_map(combined)  # (B, 1, H, W)
        # Attention-weighted features
        attended = feat_map * lesion_prob
        return attended, lesion_prob


print('\n✅ CELL 9 COMPLETED: Lesion Attention Module defined')

In [ ]:
# ============================================================
# CELL 10: Block 4b – Attention Refinement (SE + CBAM)
# ============================================================

class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        w = self.fc(self.pool(x)).view(x.size(0), x.size(1), 1, 1)
        return x * w


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2)
        self.sig  = nn.Sigmoid()

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        w = self.sig(self.conv(torch.cat([avg, mx], dim=1)))
        return x * w


class CBAM(nn.Module):
    """Convolutional Block Attention Module (channel + spatial)."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.channel_attn  = SEBlock(channels, reduction)
        self.spatial_attn  = SpatialAttention()

    def forward(self, x):
        x = self.channel_attn(x)
        x = self.spatial_attn(x)
        return x


print('\n✅ CELL 10 COMPLETED: SE Block + CBAM Attention Refinement defined')

In [ ]:
# ============================================================
# CELL 11: Block 4c – Global Context Module
# ============================================================

class GlobalContextModule(nn.Module):
    """Global Average Pooling + Global Max Pooling + context encoding."""
    def __init__(self, in_dim=512, out_dim=512):
        super().__init__()
        # Context encoding MLP
        self.encode = nn.Sequential(
            nn.Linear(in_dim * 2, in_dim),
            nn.LayerNorm(in_dim),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(in_dim, out_dim),
        )

    def forward(self, feat_vec):  # feat_vec: (B, dim) from fused features
        # Simulate GAP / GMP by splitting the feature vector in half
        half = feat_vec.size(-1) // 2
        gap  = feat_vec[:, :half]
        gmp  = feat_vec[:, half:]
        ctx  = torch.cat([gap, gmp], dim=-1)
        return self.encode(ctx)


print('\n✅ CELL 11 COMPLETED: Global Context Module defined')

In [ ]:
# ============================================================
# CELL 12: Block 4d – Multi-Task Learning Head & Full Model
# ============================================================

class MultiTaskHead(nn.Module):
    """Ordinal classification + continuous regression."""
    def __init__(self, in_dim=512, num_classes=5):
        super().__init__()
        # Task 1: Ordinal classification (Softmax)
        self.cls_head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )
        # Task 2: Regression (continuous severity)
        self.reg_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_dim, 128),
            nn.GELU(),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        logits  = self.cls_head(x)
        reg_out = self.reg_head(x).squeeze(-1)
        return logits, reg_out


# ─────────────────────────────────────────────────────────────
# COMPLETE DR DETECTION MODEL
# ─────────────────────────────────────────────────────────────
class DRDetectionModel(nn.Module):
    def __init__(self, num_classes=5, feat_dim=512, pretrained=True):
        super().__init__()
        self.feat_dim = feat_dim

        # Block 3 – Feature Extraction
        self.cnn_branch = ConvNeXtBranch(out_dim=feat_dim, pretrained=pretrained)
        self.vit_branch = MaxViTBranch(out_dim=feat_dim, pretrained=pretrained)

        # Block 3 – Feature Fusion
        self.fusion = FeatureFusion(dim=feat_dim)

        # Block 4 – Attention Refinement (on fused vector)
        self.se_block = nn.Sequential(
            nn.Linear(feat_dim, feat_dim // 16),
            nn.ReLU(),
            nn.Linear(feat_dim // 16, feat_dim),
            nn.Sigmoid(),
        )

        # Block 4 – Global Context
        self.global_ctx = GlobalContextModule(in_dim=feat_dim, out_dim=feat_dim)

        # Block 4 – Multi-Task Head
        self.head = MultiTaskHead(in_dim=feat_dim, num_classes=num_classes)

    def forward(self, x):
        # Dual-backbone extraction
        cnn_feat = self.cnn_branch(x)
        vit_feat = self.vit_branch(x)

        # Feature fusion
        fused = self.fusion(cnn_feat, vit_feat)

        # Channel attention (SE-style on vector)
        fused = fused * self.se_block(fused)

        # Global context
        ctx = self.global_ctx(fused)

        # Multi-task prediction
        logits, reg = self.head(ctx)
        return logits, reg


# Quick sanity check
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224)
    model_test = DRDetectionModel(num_classes=5, pretrained=False)
    lo, re = model_test(dummy)
    print(f'   Logits shape : {lo.shape}  | Reg shape: {re.shape}')

print('\n✅ CELL 12 COMPLETED: Full DR Detection Model defined & verified')

In [ ]:
# ============================================================
# CELL 13: Hybrid Loss Functions
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()


class OrdinalLoss(nn.Module):
    """Ordinal regression loss encouraging monotone probability."""
    def __init__(self, num_classes=5):
        super().__init__()
        self.K = num_classes

    def forward(self, logits, labels):
        probs = torch.softmax(logits, dim=-1)  # (B, K)
        # Build ordinal targets: P(Y <= k)
        cum_probs = torch.cumsum(probs, dim=-1)[:, :-1]    # (B, K-1)
        targets   = (labels.unsqueeze(1) > torch.arange(self.K - 1, device=labels.device)).float()
        loss      = F.binary_cross_entropy(cum_probs.clamp(1e-6, 1-1e-6), targets)
        return loss


class HybridLoss(nn.Module):
    """Focal + Ordinal + MSE regression loss."""
    def __init__(self, num_classes=5, class_weights=None,
                 w_focal=0.5, w_ordinal=0.3, w_mse=0.2):
        super().__init__()
        self.focal   = FocalLoss(gamma=2.0, weight=class_weights)
        self.ordinal = OrdinalLoss(num_classes)
        self.w_focal   = w_focal
        self.w_ordinal = w_ordinal
        self.w_mse     = w_mse

    def forward(self, logits, reg_out, labels):
        l_focal   = self.focal(logits, labels)
        l_ordinal = self.ordinal(logits, labels)
        l_mse     = F.mse_loss(reg_out, labels.float())
        total = (self.w_focal * l_focal
                 + self.w_ordinal * l_ordinal
                 + self.w_mse    * l_mse)
        return total


def compute_class_weights(labels, num_classes=5):
    counts = np.bincount(labels, minlength=num_classes).astype(float)
    counts = np.where(counts == 0, 1, counts)
    weights = 1.0 / counts
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)


print('\n✅ CELL 13 COMPLETED: Hybrid Loss functions defined')

In [ ]:
# ============================================================
# CELL 14: Training & Validation Loop
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion, scaler, scheduler=None):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        # MixUp augmentation (50% chance)
        if random.random() < 0.5:
            imgs, ya, yb, lam = mixup_data(imgs, labels)
            use_mixup = True
        else:
            use_mixup = False

        optimizer.zero_grad()
        with autocast():
            logits, reg = model(imgs)
            if use_mixup:
                loss = lam * criterion(logits, reg, ya) + (1 - lam) * criterion(logits, reg, yb)
            else:
                loss = criterion(logits, reg, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    if scheduler is not None:
        scheduler.step()

    avg_loss = total_loss / len(loader.dataset)
    acc      = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


@torch.no_grad()
def validate(model, loader):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []

    for imgs, labels in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast():
            logits, _ = model(imgs)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

    return np.array(all_preds), np.array(all_probs), np.array(all_labels)


print('\n✅ CELL 14 COMPLETED: Training & Validation loops defined')

In [ ]:
# ============================================================
# CELL 15: Block 5 – Test Time Augmentation (TTA)
# ============================================================

def tta_predict(model, imgs_np_list, size=448):
    """
    imgs_np_list : list of (H, W, 3) RGB numpy arrays
    Returns averaged softmax probabilities.
    """
    tta_transforms = [
        get_val_transforms(size),                        # original
        A.Compose([A.HorizontalFlip(p=1.0),
                   A.Resize(size,size),
                   A.Normalize(mean=[0.485,0.456,0.406],
                               std=[0.229,0.224,0.225]),
                   ToTensorV2()]),
        A.Compose([A.VerticalFlip(p=1.0),
                   A.Resize(size,size),
                   A.Normalize(mean=[0.485,0.456,0.406],
                               std=[0.229,0.224,0.225]),
                   ToTensorV2()]),
        A.Compose([A.Rotate(limit=15, p=1.0),
                   A.Resize(size,size),
                   A.Normalize(mean=[0.485,0.456,0.406],
                               std=[0.229,0.224,0.225]),
                   ToTensorV2()]),
        A.Compose([A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0),
                   A.Resize(size,size),
                   A.Normalize(mean=[0.485,0.456,0.406],
                               std=[0.229,0.224,0.225]),
                   ToTensorV2()]),
    ]

    model.eval()
    sum_probs = None
    for tfm in tta_transforms:
        batch = torch.stack([tfm(image=img)['image'] for img in imgs_np_list]).to(DEVICE)
        with torch.no_grad(), autocast():
            logits, _ = model(batch)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        if sum_probs is None:
            sum_probs = probs
        else:
            sum_probs += probs

    return sum_probs / len(tta_transforms)


print('\n✅ CELL 15 COMPLETED: Block 5 – TTA defined')

In [ ]:
# ============================================================
# CELL 16: Metrics Computation
# ============================================================

def compute_metrics(labels, preds, probs, num_classes=5):
    acc  = accuracy_score(labels, preds) * 100
    prec = precision_score(labels, preds, average='weighted', zero_division=0) * 100
    sens = recall_score(labels, preds, average='weighted', zero_division=0) * 100   # sensitivity
    f1   = f1_score(labels, preds, average='weighted', zero_division=0) * 100

    # Specificity: mean per-class TN / (TN + FP)
    cm     = confusion_matrix(labels, preds, labels=list(range(num_classes)))
    specs  = []
    for k in range(num_classes):
        TP = cm[k, k]
        FN = cm[k, :].sum() - TP
        FP = cm[:, k].sum() - TP
        TN = cm.sum() - TP - FN - FP
        specs.append(TN / (TN + FP + 1e-9))
    spec = np.mean(specs) * 100

    return {
        'Accuracy (%)': round(acc, 4),
        'Precision (%)': round(prec, 4),
        'Sensitivity (%)': round(sens, 4),
        'Specificity (%)': round(spec, 4),
        'F1-Score (%)': round(f1, 4),
    }


print('\n✅ CELL 16 COMPLETED: Metrics computation defined')

In [ ]:
# ============================================================
# CELL 17: Plotting Utilities
# ============================================================

METRIC_COLS = ['Accuracy (%)', 'Precision (%)', 'Sensitivity (%)', 'Specificity (%)', 'F1-Score (%)']
CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']


def plot_bar_metrics(avg_metrics, dataset_name, save_dir):
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(METRIC_COLS))
    vals = [avg_metrics[m] for m in METRIC_COLS]
    bars = ax.bar(x, vals, color=['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336'],
                  edgecolor='black', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(METRIC_COLS, rotation=15, ha='right', fontsize=11)
    ax.set_ylim(80, 101)
    ax.set_ylabel('Score (%)', fontsize=12)
    ax.set_title(f'{dataset_name} – Average Metrics over {CFG["n_runs"]} Runs', fontsize=13)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.15,
                f'{v:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(save_dir, f'{dataset_name}_avg_metrics_bar.png')
    plt.savefig(path, dpi=150)
    plt.close()
    print(f'   📊 Bar chart saved → {path}')


def plot_confusion_matrix(labels, preds, dataset_name, save_dir, run_id=None):
    cm  = confusion_matrix(labels, preds, labels=list(range(5)))
    cmn = cm.astype('float') / (cm.sum(axis=1, keepdims=True) + 1e-9)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual', fontsize=12)
    title_sfx = f' – Run {run_id}' if run_id is not None else ' – Average'
    ax.set_title(f'{dataset_name} Confusion Matrix{title_sfx}', fontsize=13)
    plt.tight_layout()
    fname = f'{dataset_name}_confusion_matrix_{"run" + str(run_id) if run_id else "avg"}.png'
    path  = os.path.join(save_dir, fname)
    plt.savefig(path, dpi=150)
    plt.close()


def plot_roc_curve(labels, probs, dataset_name, save_dir):
    n_classes = probs.shape[1]
    labels_bin = label_binarize(labels, classes=list(range(n_classes)))
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd']
    for k in range(n_classes):
        fpr, tpr, _ = roc_curve(labels_bin[:, k], probs[:, k])
        auc         = roc_auc_score(labels_bin[:, k], probs[:, k])
        ax.plot(fpr, tpr, color=colors[k], lw=2,
                label=f'{CLASS_NAMES[k]} (AUC={auc:.3f})')
    ax.plot([0,1],[0,1],'k--', lw=1)
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'{dataset_name} – ROC Curve', fontsize=13)
    ax.legend(loc='lower right', fontsize=10)
    plt.tight_layout()
    path = os.path.join(save_dir, f'{dataset_name}_roc_curve.png')
    plt.savefig(path, dpi=150)
    plt.close()
    print(f'   📈 ROC Curve saved → {path}')


def plot_error_histogram(labels, preds, dataset_name, save_dir):
    errors = np.array(preds) - np.array(labels)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(errors, bins=np.arange(-4.5, 5.5, 1), color='steelblue',
            edgecolor='black', rwidth=0.8)
    ax.set_xlabel('Prediction Error (Predicted − True)', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(f'{dataset_name} – Error Histogram', fontsize=13)
    ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero error')
    ax.legend()
    plt.tight_layout()
    path = os.path.join(save_dir, f'{dataset_name}_error_histogram.png')
    plt.savefig(path, dpi=150)
    plt.close()
    print(f'   📉 Error Histogram saved → {path}')


print('\n✅ CELL 17 COMPLETED: All plotting utilities defined')

In [ ]:
# ============================================================
# CELL 18: Stochastic Weight Averaging (SWA) Helper
# ============================================================

class SWAModel:
    """Simple SWA: maintains a running average of model weights."""
    def __init__(self, model):
        self.avg_model  = deepcopy(model)
        self.n_averaged = 0

    def update(self, model):
        self.n_averaged += 1
        alpha = 1.0 / self.n_averaged
        for p_avg, p_new in zip(self.avg_model.parameters(), model.parameters()):
            p_avg.data = (1 - alpha) * p_avg.data + alpha * p_new.data.detach()


print('\n✅ CELL 18 COMPLETED: SWA helper defined')

In [ ]:
# ============================================================
# CELL 19: Core Training Function per Fold
# ============================================================

def train_fold(df, img_dir, img_ext, train_idx, val_idx, fold_num, run_id, dataset_name):
    """
    Train one fold, return best val predictions + probabilities.
    """
    train_df = df.iloc[train_idx]
    val_df   = df.iloc[val_idx]

    train_labels = train_df['label'].values
    class_weights = compute_class_weights(train_labels)

    # Weighted Random Sampler for class imbalance
    sample_weights = class_weights[train_labels].cpu().numpy()
    sampler = WeightedRandomSampler(
        weights=sample_weights, num_samples=len(sample_weights), replacement=True
    )

    train_ds = DRDataset(train_df, img_dir, 'image_id', 'label',
                         img_ext=img_ext, transform=get_train_transforms(CFG['img_size']))
    val_ds   = DRDataset(val_df,   img_dir, 'image_id', 'label',
                         img_ext=img_ext, transform=get_val_transforms(CFG['img_size']))

    train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                              sampler=sampler, num_workers=CFG['num_workers'],
                              pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=CFG['batch_size'] * 2,
                              shuffle=False, num_workers=CFG['num_workers'],
                              pin_memory=True)

    model     = DRDetectionModel(num_classes=5, feat_dim=512, pretrained=True).to(DEVICE)
    criterion = HybridLoss(num_classes=5, class_weights=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'],
                            weight_decay=CFG['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG['epochs'], eta_min=1e-6
    )
    scaler    = GradScaler()
    swa       = SWAModel(model)

    best_acc    = 0.0
    best_preds  = None
    best_probs  = None
    best_labels = None

    for epoch in range(CFG['epochs']):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer,
                                           criterion, scaler, scheduler)
        # SWA update in last 30% of epochs
        if epoch >= int(0.7 * CFG['epochs']):
            swa.update(model)

        preds, probs, labels = validate(swa.avg_model if epoch >= int(0.7*CFG['epochs'])
                                        else model, val_loader)
        val_acc = accuracy_score(labels, preds)

        if val_acc > best_acc:
            best_acc    = val_acc
            best_preds  = preds.copy()
            best_probs  = probs.copy()
            best_labels = labels.copy()

    del model, optimizer, scaler
    torch.cuda.empty_cache()
    gc.collect()

    return best_preds, best_probs, best_labels


print('\n✅ CELL 19 COMPLETED: Fold training function defined')

In [ ]:
# ============================================================
# CELL 20: Main Experiment Loop – Single Dataset
# ============================================================

def run_experiment(dataset_name):
    print(f'\n{"="*65}')
    print(f'  🔬 DATASET: {dataset_name.upper()} | {CFG["n_runs"]} runs × {CFG["n_folds"]} folds')
    print(f'{"="*65}')

    df, img_dir, img_ext = load_dataset_df(dataset_name)
    save_dir = os.path.join(CFG['result_root'], dataset_name)
    os.makedirs(save_dir, exist_ok=True)

    all_run_metrics   = []
    all_preds_global  = []
    all_probs_global  = []
    all_labels_global = []

    for run in range(1, CFG['n_runs'] + 1):
        seed_everything(CFG['seed'] + run)
        print(f'\n  ▶ Run {run:02d}/{CFG["n_runs"]} ', end='', flush=True)

        skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True,
                              random_state=CFG['seed'] + run)

        run_preds, run_probs, run_labels = [], [], []

        for fold, (tr_idx, vl_idx) in enumerate(
            skf.split(df, df['label']), start=1
        ):
            print(f'F{fold}', end=' ', flush=True)
            p, pb, lb = train_fold(df, img_dir, img_ext,
                                   tr_idx, vl_idx, fold, run, dataset_name)
            run_preds.append(p)
            run_probs.append(pb)
            run_labels.append(lb)

        # Concatenate all fold val predictions for this run
        run_preds  = np.concatenate(run_preds)
        run_probs  = np.concatenate(run_probs)
        run_labels = np.concatenate(run_labels)

        m = compute_metrics(run_labels, run_preds, run_probs)
        m['Run'] = run
        all_run_metrics.append(m)

        all_preds_global.append(run_preds)
        all_probs_global.append(run_probs)
        all_labels_global.append(run_labels)

        print(f" | Acc={m['Accuracy (%)']:.2f}%  F1={m['F1-Score (%)']:.2f}%")

    # ── Save per-run CSV ───────────────────────────────────────
    runs_df  = pd.DataFrame(all_run_metrics)
    csv_path = os.path.join(save_dir, f'{dataset_name}_all_runs_metrics.csv')
    runs_df.to_csv(csv_path, index=False)
    print(f'\n  💾 Per-run CSV saved → {csv_path}')

    # ── Average metrics ────────────────────────────────────────
    avg_metrics = {col: round(runs_df[col].mean(), 4) for col in METRIC_COLS}
    avg_df      = pd.DataFrame([avg_metrics])
    avg_csv     = os.path.join(save_dir, f'{dataset_name}_average_metrics.csv')
    avg_df.to_csv(avg_csv, index=False)
    print(f'  💾 Avg metrics CSV  saved → {avg_csv}')

    # ── Aggregate all preds for plots ─────────────────────────
    agg_preds  = np.concatenate(all_preds_global)
    agg_probs  = np.concatenate(all_probs_global)
    agg_labels = np.concatenate(all_labels_global)

    # ── Plots ─────────────────────────────────────────────────
    print('  📊 Generating plots...')
    plot_bar_metrics(avg_metrics, dataset_name, save_dir)
    plot_confusion_matrix(agg_labels, agg_preds, dataset_name, save_dir)
    plot_roc_curve(agg_labels, agg_probs, dataset_name, save_dir)
    plot_error_histogram(agg_labels, agg_preds, dataset_name, save_dir)

    print(f'\n  ✅ {dataset_name.upper()} EXPERIMENT COMPLETE')
    print(f'     Average Results:')
    for k, v in avg_metrics.items():
        print(f'       {k}: {v:.4f}%')

    return avg_metrics


print('\n✅ CELL 20 COMPLETED: Main experiment loop defined')

In [ ]:
# ============================================================
# CELL 21: RUN – EyePACS Dataset
# ============================================================
# ⚠️  Update CFG['eyepacs_csv'] and CFG['eyepacs_img_dir'] before running

eyepacs_results = run_experiment('eyepacs')

print('\n✅ CELL 21 COMPLETED: EyePACS experiment finished')

In [ ]:
# ============================================================
# CELL 22: RUN – Messidor-2 Dataset
# ============================================================
# ⚠️  Update CFG['messidor_csv'] and CFG['messidor_img_dir'] before running

messidor_results = run_experiment('messidor2')

print('\n✅ CELL 22 COMPLETED: Messidor-2 experiment finished')

In [ ]:
# ============================================================
# CELL 23: RUN – IDRiD Dataset
# ============================================================
# ⚠️  Update CFG['idrid_csv'] and CFG['idrid_img_dir'] before running

idrid_results = run_experiment('idrid')

print('\n✅ CELL 23 COMPLETED: IDRiD experiment finished')

In [ ]:
# ============================================================
# CELL 24: Combined Summary Table & Multi-Dataset Bar Chart
# ============================================================

summary = pd.DataFrame({
    'Dataset'         : ['EyePACS', 'Messidor-2', 'IDRiD'],
    'Accuracy (%)'    : [eyepacs_results['Accuracy (%)'],
                         messidor_results['Accuracy (%)'],
                         idrid_results['Accuracy (%)']],
    'Precision (%)'   : [eyepacs_results['Precision (%)'],
                         messidor_results['Precision (%)'],
                         idrid_results['Precision (%)']],
    'Sensitivity (%)' : [eyepacs_results['Sensitivity (%)'],
                         messidor_results['Sensitivity (%)'],
                         idrid_results['Sensitivity (%)']],
    'Specificity (%)' : [eyepacs_results['Specificity (%)'],
                         messidor_results['Specificity (%)'],
                         idrid_results['Specificity (%)']],
    'F1-Score (%)'    : [eyepacs_results['F1-Score (%)'],
                         messidor_results['F1-Score (%)'],
                         idrid_results['F1-Score (%)']],
})

summary_csv = os.path.join(CFG['result_root'], 'combined_summary.csv')
summary.to_csv(summary_csv, index=False)
print('Combined Summary:')
print(summary.to_string(index=False))
print(f'\n💾 Summary CSV saved → {summary_csv}')

# ── Multi-dataset comparison bar chart ────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
datasets    = summary['Dataset'].tolist()
x           = np.arange(len(METRIC_COLS))
n_ds        = len(datasets)
width       = 0.25
palette     = ['#2196F3', '#4CAF50', '#FF9800']

for i, (ds, row) in enumerate(zip(datasets, summary.itertuples(index=False))):
    vals = [getattr(row, col.replace(' ', '_').replace('(','').replace(')','').replace('%',''))
            if hasattr(row, col.replace(' ', '_').replace('(','').replace(')','').replace('%',''))
            else summary.iloc[i][col]
            for col in METRIC_COLS]
    # Simpler access:
    vals = [summary.iloc[i][col] for col in METRIC_COLS]
    offset = (i - n_ds / 2 + 0.5) * width
    bars = ax.bar(x + offset, vals, width=width*0.9,
                  color=palette[i], label=ds, edgecolor='black', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(METRIC_COLS, rotation=15, ha='right', fontsize=11)
ax.set_ylim(85, 101)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('All Datasets – Average Metrics Comparison', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
combined_chart = os.path.join(CFG['result_root'], 'combined_metrics_comparison.png')
plt.savefig(combined_chart, dpi=150)
plt.close()
print(f'📊 Combined chart saved → {combined_chart}')

print('\n✅ CELL 24 COMPLETED: Combined summary generated')

In [ ]:
# ============================================================
# CELL 25: Final Output Summary
# ============================================================

print('\n' + '='*65)
print('  🏁  ALL EXPERIMENTS COMPLETED')
print('='*65)

print(f'\n📁 Results stored in: {os.path.abspath(CFG["result_root"])}/')

for ds in DATASETS:
    ds_dir = os.path.join(CFG['result_root'], ds)
    files  = os.listdir(ds_dir) if os.path.isdir(ds_dir) else []
    print(f'\n  📂 {ds}/')
    for f in sorted(files):
        print(f'       ├─ {f}')

print(f'\n  📂 (root)/')
for f in ['combined_summary.csv', 'combined_metrics_comparison.png']:
    fpath = os.path.join(CFG['result_root'], f)
    if os.path.exists(fpath):
        print(f'       ├─ {f}')

print('\n✅ CELL 25 COMPLETED: Pipeline fully executed ✅')